In [12]:
from cgra import *
from kernels import *
from scripts import sat_to_csv
import random


In [13]:
kernel_name = "benchmarks/compigra/yuxuan/relu_majo"
version = "_unroll_mod"

In [14]:
# Global variables
CGRB_N_ROWS = 4
CGRA_N_COLS = 4
# Adress
first_addr = 20000

In [15]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [16]:
# Data
def configMemory(data, data_size):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------
    # &Im   &Im    data_size_addr   &Im 
    # &Im   &Im    &Im              &Im

    config_vals_col0 = [first_addr, first_addr]
    config_vals_col1 = [first_addr, first_addr]
    config_vals_col2 = [first_addr, int(data_size/4)]
    config_vals_col3 = [first_addr, first_addr]
    
    addr_config_loads_col0 = 0
    kernel_add_memory_region(kernel_name, addr_config_loads_col0, config_vals_col0, version=version)
    addr_config_loads_col1 = addr_config_loads_col0 + len(config_vals_col0)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col1, config_vals_col1, version=version)
    addr_config_loads_col2 = addr_config_loads_col1 + len(config_vals_col1)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col2, config_vals_col2, version=version)
    addr_config_loads_col3 = addr_config_loads_col2 + len(config_vals_col2)*4
    kernel_add_memory_region(kernel_name, addr_config_loads_col3, config_vals_col3, version=version)
    # Load data
    kernel_add_memory_region(kernel_name, first_addr, data, version=version)
    # Config data address for direct loads
    load_addrs = [addr_config_loads_col0, addr_config_loads_col1, addr_config_loads_col2, addr_config_loads_col3]
    return load_addrs

In [17]:
def runKernel(load_addrs, max_it=1000, printVal=1):
    # Run kernel
    run(kernel_name, pr=["ROUT","INST"], load_addrs=load_addrs, version=version, limit=max_it)

In [18]:
def getResult(first_addr_C, end_addr_C, vlen):
    result = [0 for _ in range(vlen)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [19]:
def relu_cpu(image, vlen):
    expected_res = [0 for _ in range(vlen)]
    for i in range(vlen):
        if image[i] > 0:
            expected_res[i] = image[i]
        else:
            expected_res[i] = 0
    return expected_res

In [20]:
# Test dimensions
IMAGE_SIZE = 32*32
image = [random.randint(-20, 20) for _ in range(IMAGE_SIZE)]
n_negs = sum(1 for x in image if x < 0)


load_addrs = configMemory(image, IMAGE_SIZE)

In [21]:
runKernel(load_addrs, max_it=200000, printVal=0)

Instr =  0 ( 0 )
[   0,    0,    0,    0]    [NOP , NOP , NOP , NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , NOP , NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , NOP , NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , SADD R0  ZERO ZERO, NOP ]    
-------
Instr =  1 ( 1 )
[   0, 20000,    0, 20000]    [NOP , LWD R0  4, NOP , LWD R0  4]    
[20000,    0, 20000, 20000]    [LWD R0  4, NOP , LWD R0  4, LWD R0  4]    
[20000, 20000,    0,    0]    [LWD R0  4, LWD R0  4, NOP , NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , SADD ROUT  ZERO ZERO, NOP ]    
-------
Instr =  2 ( 2 )
[   0, 20000,    0, 20000]    [NOP , NOP , NOP , NOP ]    
[20000,    0,  256, 20000]    [NOP , NOP , LWD R1  4, NOP ]    
[20000, 20000,    0,    0]    [NOP , NOP , NOP , NOP ]    
[   0,    0,    0,    0]    [NOP , NOP , NOP , NOP ]    
-------
Instr =  3 ( 3 )
[   0, 20000,    0, 20000]    [NOP , NOP , NOP , NOP ]    
[20000,    0,  256, 20000]    [NOP , NOP , SADD ROUT  ROUT  ZERO, NOP ]    
[20000

In [22]:
# Get result from CGRA
end_addr = first_addr + IMAGE_SIZE*4
result = getResult(first_addr, end_addr, IMAGE_SIZE)

# Get cpu output
expected_res = relu_cpu(image, IMAGE_SIZE)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
        print("Error at index " + str(i) + ": expected " + str(expected_res[i]) + ", got " + str(result[i]))
if errors > 0:
    print("N negs: " + str(n_negs))
    print("Err: " + str(errors))
    print("CGRA: " + str(result))
    print("CPU:  " + str(expected_res))
else:
    print("OK")



OK
